# PMQC — Perfil Exploratório

Análise do Programa de Monitoramento da Qualidade dos Combustíveis (2016–2026).

**Fonte:** `data/trusted/pmqc/pmqc.parquet` (8.4M ensaios, 48k CNPJs)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path.cwd().parents[2]
TRUSTED = REPO / 'data' / 'trusted'

df = pd.read_parquet(TRUSTED / 'pmqc' / 'pmqc.parquet')
print(f'Shape: {df.shape}')
print(f'Periodo: {df["data_coleta"].min()} a {df["data_coleta"].max()}')
df.head()

## 1. Distribuição por Grupo de Produto

In [ ]:
produto_counts = df['grupo_produto'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
produto_counts.plot.barh(ax=ax)
ax.set_xlabel('Ensaios')
ax.set_title('Ensaios por Grupo de Produto')
plt.tight_layout()
plt.show()
print(produto_counts)

## 2. Evolução Temporal

In [ ]:
df['ano_mes'] = df['data_coleta'].dt.to_period('M')
mensal = df.groupby('ano_mes').size()

fig, ax = plt.subplots(figsize=(14, 4))
mensal.plot(ax=ax)
ax.set_ylabel('Ensaios/m\u00eas')
ax.set_title('Volume de Ensaios PMQC por M\u00eas (2016\u20132026)')
plt.tight_layout()
plt.show()

## 3. Taxa de Não-Conformidade

In [ ]:
nc = df.groupby('ano_mes')['conforme'].apply(lambda x: (~x).mean() * 100)

fig, ax = plt.subplots(figsize=(14, 4))
nc.plot(ax=ax, color='red')
ax.set_ylabel('% N\u00e3o-Conforme')
ax.set_title('Taxa de N\u00e3o-Conformidade Mensal (todos os produtos)')
ax.axhline(nc.mean(), ls='--', color='gray', label=f'M\u00e9dia: {nc.mean():.3f}%')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Não-Conformidade por UF

In [ ]:
nc_uf = df.groupby('uf')['conforme'].apply(lambda x: (~x).mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
nc_uf.plot.barh(ax=ax, color='coral')
ax.set_xlabel('% N\u00e3o-Conforme')
ax.set_title('Taxa de N\u00e3o-Conformidade por UF')
plt.tight_layout()
plt.show()
print(nc_uf.head(10))

## 5. Não-Conformidade por Produto

In [ ]:
nc_prod = df.groupby(['grupo_produto', 'ano_mes'])['conforme'].apply(lambda x: (~x).mean() * 100).reset_index()
nc_prod.columns = ['grupo_produto', 'ano_mes', 'pct_nc']

fig, ax = plt.subplots(figsize=(14, 4))
for prod, grp in nc_prod.groupby('grupo_produto'):
    ax.plot(grp['ano_mes'].astype(str), grp['pct_nc'], label=prod)
ax.set_ylabel('% N\u00e3o-Conforme')
ax.set_title('N\u00e3o-Conformidade por Produto ao Longo do Tempo')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Distribuição Geográfica (Amostras por UF)

In [ ]:
amostras_uf = df.groupby('uf')['cnpj'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
amostras_uf.plot.barh(ax=ax)
ax.set_xlabel('CNPJs \u00fanicos amostrados')
ax.set_title('Postos Amostrados por UF (2016\u20132026)')
plt.tight_layout()
plt.show()
print(f'Total CNPJs: {df["cnpj"].nunique()}')
print(f'Top 5 UFs: {amostras_uf.head().to_dict()}')

## 7. Cruzamento com Cadastro de Revendas

In [ ]:
cad = pd.read_parquet(TRUSTED / 'cadastro-revendas-combustiveis' / 'revendas.parquet')
pmqc_cnpj = set(df['cnpj'].dropna().unique())
cad_cnpj = set(cad['cnpj'].dropna().unique())

overlap = pmqc_cnpj & cad_cnpj
only_pmqc = pmqc_cnpj - cad_cnpj
only_cad = cad_cnpj - pmqc_cnpj

print(f'CNPJs no PMQC: {len(pmqc_cnpj):,}')
print(f'CNPJs no Cadastro: {len(cad_cnpj):,}')
print(f'Em comum: {len(overlap):,} ({len(overlap)/len(pmqc_cnpj)*100:.1f}% dos PMQC)')
print(f'So no PMQC (nao no cadastro): {len(only_pmqc):,}')
print(f'So no Cadastro (nunca amostrados): {len(only_cad):,} ({len(only_cad)/len(cad_cnpj)*100:.1f}%)')

## 8. Postos no PMQC que NÃO constam no Cadastro

In [ ]:
postos_fora = df[df['cnpj'].isin(only_pmqc)].drop_duplicates(subset=['cnpj'])
print(f'Postos amostrados pelo PMQC que NAO constam no cadastro: {len(postos_fora)}')
print(f'Distribui\u00e7\u00e3o por UF:')
print(postos_fora['uf'].value_counts().head(10))